**File:** technical_memory.ipynb  
**Description:** Running log of the *why* behind Discord Status Updater: architectural decisions, rejected alternatives, gotchas and wrong assumptions.  
**Author:** Jose-Jorge HERNANDEZ  
**Company:** Parlee Conseiller, Inc.  
**Date:** 2026-10-09  
**Last edit date:** 2026-10-09  
**Version:** 1.1.0

# Technical Memory

Append a new dated section for every notable technical decision or gotcha. Do not rewrite earlier entries. *What* changed lives in `CHANGELOG.md`; bugs live in `docs/BUGS.md`; this notebook records the reasoning.

## 2026-10-09 - Automatic app detection (v2.0.0 design)

### Context
The app only supported manual presets. The goal: detect running dev tools (VS Code, Visual Studio, Wave Terminal...) and switch the Discord activity on its own. Discord's own detection only recognises games, so this has to be done by the app.

### Decisions

**1. Polling, not WMI/ETW process events.** A 3 s poll of `Process.GetProcessesByName` for only the names that rules mention is cheap, needs no elevation, and also gives us window titles. WMI `Win32_ProcessStartTrace` needs admin; ETW is overkill. Rejected for simplicity.

**2. Rule -> preset reference, not an embedded presence.** A `DetectionRule` carries only matching data (process names, title regex, optional Application ID) and points at a preset by name. The preset editor is reused and the presence fields are not duplicated in two places.

**3. One Discord Application ID per rule is optional.** Rich Presence shows *"Playing <Application name>"*, and the name comes from the Discord Application, not from our text. To get a card that literally says "Visual Studio Code", a rule needs its own Application ID. So each rule may override the default ID, and the RPC service reconnects when the effective ID changes. Rules without an override use the default ID and rely on Details/State/images.

**4. Debounce and rate-limit.** A new winner must be seen on two consecutive polls before switching (tools spawn short-lived helper processes and windows flicker). Presence text updates are limited to one per 15 s because Discord rate-limits `SET_ACTIVITY`.

**5. Window titles are optional personal data.** `{file}` / `{workspace}` tokens come from `MainWindowTitle`. A *Share window titles* setting blanks them, because filenames can be sensitive on a public profile.

**6. Pure logic is kept out of WPF types** (`TitleParser`, `TemplateEngine`, `RuleMatcher`) so it can be unit-tested without a UI thread.

### Title parsing example

VS Code titles look like `file.cs - folder - Visual Studio Code`. The last segment is the app label and is dropped; the first remaining segment is the file and the last remaining one the workspace. The sketch below mirrors `TitleParser.Parse` so the rule can be checked in isolation.

In [ ]:
import re

def parse(title):
    # Split on hyphen / en dash / em dash surrounded by spaces.
    parts = [p.strip() for p in re.split(r"\s[-\u2013\u2014]\s", title) if p.strip()]
    if len(parts) > 1:
        parts = parts[:-1]          # last segment is the application label
    if len(parts) == 1:
        return "", parts[0]         # only a workspace
    return parts[0], parts[-1] if parts else ("", "")

print(parse("main.cs - discord_status_updater - Visual Studio Code"))
print(parse("discord_status_updater - Visual Studio Code"))

## 2026-10-09 - Gotchas hit while building 2.0.0

**Embedded defaults were never read.** `default_presets.json` is a WPF `Resource`, so it only exists inside the assembly. The original `File.Exists(BaseDirectory\Resources\...)` was always false, so a fresh install had *no presets at all* and nobody noticed because the author already had a `presets.json`. Found because the new UI test run showed `LastPresetName = "Coding in VS Code"` from a clean data folder. Lesson: when a feature depends on shipped data, test from an empty profile.

**`%APPDATA%` is not honoured.** My first "isolated" UI test wrote to the real profile because `Environment.SpecialFolder.ApplicationData` goes through the shell known-folder API. Added `DISCORD_STATUS_UPDATER_DATA` and cleaned up the one file created. Assumption that was wrong: that overriding the env var would redirect .NET folder lookups.

**Switching Application IDs raises events from the dead client.** `DiscordRpcClient.Dispose()` fires `OnClose`; forwarding it made the UI claim "Disconnected" mid-switch. Handlers now compare the sender client with the current one.

**Title parsing is inherently ambiguous.** ` - ` appears in file names and as the separator. Chose predictable behaviour (first = file, last = workspace) over heuristics, documented the limit, and kept `{title}` as the escape hatch.

**Wave Terminal exposes no window title** (observed with 5 running processes), so its shipped preset uses fixed text only.

### Why the debounce is "2 scans", checked by a test

Focus changes and helper windows make single-scan results flicker. Two consecutive identical scans (~6 s at the default interval) was the smallest value that makes a one-scan blip invisible. The sketch below reproduces the rule used in `AutoDetectService.ScanOnce` and is covered by `AutoDetectServiceTests`.

In [ ]:
REQUIRED = 2
def debounce(scans):
    committed, pending, count, events = None, None, 0, []
    for key in scans:
        if key == committed:
            pending, count = None, 0
            continue
        if key == pending:
            count += 1
        else:
            pending, count = key, 1
        if count >= REQUIRED:
            committed, pending, count = key, None, 0
            events.append(key)
    return events

print(debounce(['code', None, 'code', 'code', 'code']))   # a one-scan blip is ignored -> ['code']
print(debounce(['code', 'code', None, None]))            # app closes -> ['code', None]
